# Hello, Memory — the low-level API

`topic()` and `library()` are conveniences built on `Memory`, which is the layer the
README's 30-second tour uses. Reach for it when you want to control ids and metadata
yourself instead of handing over files.

**Needs:** Ollama with `nomic-embed-text`.

In [1]:
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

from slim_llm_memory import Memory, Embedder

mem = Memory(ROOT / ".hello_nb" / "memory", Embedder.ollama("nomic-embed-text"))
mem.upsert([
    {"id": "doc1", "text": "how to set up nginx",       "meta": {"kind": "note"}},
    {"id": "doc2", "text": "milch kaufen",              "meta": {"kind": "shopping"}},
    {"id": "doc3", "text": "how to configure nginx",    "meta": {"kind": "note"}},
])

{'added': 3, 'updated': 0, 'skipped': 0, 'embed_calls': 1}

`upsert` is incremental: it hashes each text and only embeds the ones it has not
seen. Re-running the cell embeds nothing.

## Search

`kinds` filters on `meta.kind`, so the shopping list stays out of the way.

In [2]:
mem.search("nginx tutorial", k=3, kinds={"note"})

[Hit(id='doc1', score=0.8623174428939819, text='how to set up nginx', meta={'kind': 'note'}),
 Hit(id='doc3', score=0.8519346117973328, text='how to configure nginx', meta={'kind': 'note'})]

## Near-duplicates

`find_duplicates` clusters by cosine similarity. `doc1` and `doc3` say almost the same
thing, so they cluster; the shopping item does not.

In [3]:
mem.find_duplicates(threshold=0.86)

[['doc1', 'doc3']]

## Persistence

`flush()` writes atomically — a new versioned manifest, then a rename. If the process
dies mid-flush the previous version is still the one that loads.

In [4]:
mem.flush()
mem.stats()

{'items': 3,
 'items_open': 3,
 'tombstones': 0,
 'embedder': 'ollama:nomic-embed-text',
 'embed_dim': 768,
 'version': 1,
 'dirty': False,
 'file_age_seconds': 0,
 'needs_compaction': False,
 'counters': {'embed.calls': 1,
  'embed.items': 3,
  'upsert.added': 3,
  'search.calls': 1},
 'embed_errors_1h': 0,
 'embed_errors_24h': 0,
 'llm_errors_1h': 0,
 'llm_errors_24h': 0,
 'slow_queries': [{'ts': 1788983915.8990564, 'op': 'embed', 'ms': 1317.29},
  {'ts': 1788983916.3656566, 'op': 'search', 'ms': 458.44}],
 'uptime_seconds': 1,
 'buffer_cap': 50,
 'slow_threshold_ms': 100}